# Scrapear una temporada pasada de Primera Nacional (36 equipos)

Parte del **fixture** de la temporada (todos los partidos con su `event_id`) y baja cada partido **una sola vez**, en la carpeta del equipo local (`data/temporadas/{año}/rivals/{local}/`). Si un partido ya tiene carpeta (por ejemplo los de Chaco en `partidos/`), completa esa misma.

- Es incremental: si se corta (ban, captcha, apagaste la PC), volvé a correr la sección 3 y sigue donde quedó.
- Sin heatmaps tarda ~26 s por partido (≈4 h una temporada). Con heatmaps, ≈12 h y bastante más riesgo de ban: se pueden completar después en otra corrida.
- Antes de bajar, la sección 2 avisa los equipos que no están registrados en `config.py`: conviene sumarlos a `RIVALS_<año>` para que el slug de las carpetas quede fijo.

In [1]:
import os, sys
%load_ext autoreload
%autoreload 2
if os.path.basename(os.getcwd()) in ('primera_nacional', 'liga_profesional'):
    os.chdir('..')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd(); sys.path.insert(0, PROJECT_ROOT)

import config
from src.scraping import temporada_scraper as T

## 1. Configuración

In [2]:
# ============================================================
# CONFIGURAR ACÁ
SEASON      = '2025'   # tiene que estar en config.SEASON_IDS
HEATMAPS    = True    # True = +22 requests por partido (≈3x más lento)
HEADLESS    = False    # False siempre: con headless SofaScore pide captcha más seguido
RETRY_EMPTY = True    # True = reintentar endpoints que SofaScore no había publicado
LIMITE      = None     # ej. 3 para una prueba corta; None = toda la temporada
# ============================================================
print('season_id:', config.SEASON_IDS.get(SEASON))

season_id: 71009


## 2. Plan (baja el fixture si no está, no scrapea partidos)

Si aparecen equipos **sin registrar**, agregalos a `RIVALS_<año>` en `config.py` (nombre de slug con guiones bajos y su `team_id`) y volvé a correr esta celda.

In [3]:
plan = T.plan_temporada(SEASON)

  fecha  1: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  2: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  3: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  4: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  5: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  6: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  7: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  8: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha  9: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 10: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 11: 19 partidos (18 jugados, 1 pendientes/otros)
  fecha 12: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 13: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 14: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 15: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 16: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 17: 18 partidos (18 jugados, 0 pendientes/otros)
  fecha 18: 18 partidos (18 jug

## 3. Scrapear

In [4]:
stats = T.scrape_temporada(SEASON, headless=HEADLESS, include_heatmaps=HEATMAPS,
                           retry_empty=RETRY_EMPTY, limite=LIMITE)

Temporada 2025: 612 partidos terminados en el fixture · 36 ya tienen carpeta · 576 nuevos
  [1/612] colon/2025-02-07_vs_temperley_h
  [2/612] arsenal_de_sarandi/2025-02-07_vs_all_boys_h
    [challenge] renuevo sofa_captcha (intento 1/3)...
    [challenge] renuevo sofa_captcha (intento 1/3)...
  [3/612] nueva_chicago/2025-02-07_vs_san_telmo_h
  [4/612] partidos/ya está: 2025-02-07_vs_estudiantes_caseros_h
  [5/612] almagro/2025-02-08_vs_san_martin_tucuman_h
  [6/612] colegiales/2025-02-08_vs_alvarado_h
  [7/612] los_andes/2025-02-08_vs_patronato_h
  [8/612] quilmes/2025-02-08_vs_gimnasia_tiro_salta_h
  [9/612] san_miguel/2025-02-08_vs_tristan_suarez_h
  [10/612] gimnasia_y_esgrima_mendoza/2025-02-08_vs_estudiantes_de_rio_cuarto_h
  [11/612] central_norte/2025-02-09_vs_talleres_remedios_h
  [12/612] gimnasia_jujuy/2025-02-09_vs_defensores_de_belgrano_h
  [13/612] racing_de_cordoba/2025-02-09_vs_deportivo_maipu_h
  [14/612] almirante_brown/2025-02-09_vs_deportivo_moron_h
  [15/612] club_m

## 4. Tabla de posiciones de las dos zonas

In [5]:
import io, contextlib
from src.scraping.standings_scraper import fetch_standings, fetch_standings_zona_b
try:
    with contextlib.redirect_stdout(io.StringIO()):   # los emoji revientan en cp1252
        fetch_standings(season=SEASON, headless=False)
        fetch_standings_zona_b(season=SEASON, headless=False)
except Exception as e:
    print('Aviso al bajar posiciones:', repr(e))
import pandas as pd
for f in (config.standings_file(SEASON), config.standings_zona_b_file(SEASON)):
    if os.path.exists(f):
        t = pd.read_csv(f); print(os.path.basename(f), len(t), 'equipos · líder:', t.iloc[0]['team_name'])

standings.csv 18 equipos · líder: Deportivo Madryn
standings_zona_b.csv 18 equipos · líder: Gimnasia y Esgrima Mendoza


## 5. Sumar la temporada a la base de jugadores y controlar

In [6]:
from src.jugadores import construir as C
C.construir([SEASON])
C.verificar(SEASON)
dups = C.posibles_duplicados()

[2025] 612 partidos: 34 sin cambios, 578 a leer


c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\src\jugadores\construir.py:570: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  jp = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()



Base en c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\jugadores
  partidos          : 1225
  equipos           : 72 (temporada × equipo)
  jugadores         : 1837
  jugador_partido   : 48898 filas en 2 temporadas
  jugador_temporada : 2366 filas
[2025] partidos 612 (con lineups 612) · jugadores 1267 · máx. partidos de un jugador 34 (rondas 34)
  en cancha por equipo y partido (mediana): 11.00 · filas con datos completos: 97.0%
  OK
11 posibles duplicados (revisar a mano):
 temporada              equipo     nombre_corto  player_id    nombre_completo nacimiento  partidos primer_partido ultimo_partido
      2025     almirante_brown          L. Vega     988200       Leandro Vega 1996-05-27         3     2025-02-09     2025-05-03
      2025     almirante_brown          L. Vega    2143378         Lucas Vega 2004-05-19         8     2025-07-13     2025-10-05
      2026            all_boys    N. Barrientos    1111801 Nicolas Barrientos 1995-04-05        16     2026-07-05   